# Steam Games — Final Project

**FIT@HCMUS · Introduction to Data Science 4.0**

| | |
|---|---|
| **Dataset** | Steam Games (March 2025) — Kaggle |
| **File** | `games_march2025_cleaned.csv` (~450 MB) |
| **Nhom** | *(Dien ten nhom)* |
| **Thanh vien** | *(Member 1)* - *(Member 2)* - *(Member 3)* |

---

## Muc luc

1. [Thu thap du lieu](#1-thu-thap-du-lieu)
2. [Kham pha du lieu (EDA)](#2-kham-pha-du-lieu-eda)
   - 2.1 Tong quan dataset
   - 2.2 Y nghia tung cot (Data Dictionary)
   - 2.3 Kieu du lieu
   - 2.4 Missing values
   - 2.5 Phan tich cot so (Numerical)
   - 2.6 Phan tich cot phan loai (Categorical)
   - 2.7 Phan phoi theo thoi gian
   - 2.8 Tuong quan giua cac bien so
3. [Dat cau hoi co y nghia](#3-dat-cau-hoi-co-y-nghia)
4. [Tien xu ly & Phan tich](#4-tien-xu-ly--phan-tich)
5. [Phan anh (Reflection)](#5-phan-anh-reflection)
6. [Tai lieu tham khao](#6-tai-lieu-tham-khao)

## Setup — Import thu vien & Load du lieu

In [ ]:
# Core
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import missingno as msno

# Style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (12, 5),
    'font.size': 12,
})

# Project utils
import sys
sys.path.append(os.path.join('..', '..'))
from src.data_loader import load_data, get_basic_info

# Thu muc luu bieu do
FIG_DIR = os.path.join('..', '..', 'reports', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

print('Import xong tat ca thu vien.')

In [ ]:
# Load du lieu
DATA_PATH = os.path.join('..', '..', 'data', 'raw', 'games_march2025_cleaned.csv')

df = load_data(DATA_PATH)
print(f'Shape: {df.shape[0]:,} hang x {df.shape[1]} cot')
df.head()

---

## 1. Thu thap du lieu

> Tra loi 3 cau hoi theo yeu cau de bai.

### 1.1 Chu de & Nguon goc du lieu

**Chu de:**
Bo du lieu ve cac tua game tren nen tang **Steam** (Valve Corporation) — nen tang phan phoi game lon nhat the gioi. Du lieu duoc thu thap den thang 3 nam 2025, chua thong tin cua hon 50,000 tua game.

**Nguon:**
[Kaggle — Steam Games Dataset (March 2025)](https://www.kaggle.com/)
*(Cap nhat link chinh xac sau khi nhom xac nhan)*

**Cac truong thong tin chinh bao gom:**
- Ten game, nha phat trien, nha phat hanh
- Ngay phat hanh, gia ban (USD)
- So danh gia tich cuc / tieu cuc
- The loai (genres), tags
- Nen tang ho tro (Windows, Mac, Linux)
- Ngon ngu ho tro, so DLC, so thanh tich (achievement)

### 1.2 Giay phep su dung du lieu

*(Dien sau khi kiem tra trang Kaggle cua dataset)*

| Thong tin | Chi tiet |
|---|---|
| **License** | *(vi du: CC0 1.0 Universal / ODbL 1.0 / ...)* |
| **Cho phep su dung hoc thuat?** | *(Co / Khong / Can doc them dieu khoan)* |
| **Link license** | *(URL day du)* |
| **Han che su dung** | *(Co han che gi khong?)* |

### 1.3 Cach tac gia thu thap du lieu

*(Dien sau khi doc mo ta dataset tren Kaggle — phan 'About Dataset')*

**Phuong phap thu thap (placeholder — can kiem chung):**

1. **Lay danh sach app_id**: Su dung Steam Web API endpoint `ISteamApps/GetAppList` de lay toan bo danh sach App ID tren Steam.
2. **Lay thong tin chi tiet**: Voi moi `app_id`, goi tiep API `IStoreService/GetAppDetails` hoac scrape trang `store.steampowered.com/app/{app_id}`.
3. **Lam sach so bo**: Tac gia da loc bot cac app khong phai game (phan mem, DLC standalone) va xu ly mot so truong du lieu truoc khi dang len Kaggle.
4. **Snapshot**: Du lieu duoc chup lai vao thang 3/2025.

**Ghi chu**: Can doc ky mo ta goc tren Kaggle de co thong tin chinh xac.

---

## 2. Kham pha du lieu (EDA)

> Ket hop voi tien xu ly. Tra loi day du cac cau hoi theo checklist de bai.

### 2.1 Tong quan dataset

In [ ]:
# In thong tin co ban: so hang, so cot, bo nho, duplicate, kieu du lieu, missing
get_basic_info(df)

In [ ]:
# Xem 5 hang dau de nam bat cau truc du lieu
df.head()

In [ ]:
# Xem 5 hang cuoi — kiem tra xem co hang rong cuoi file khong
df.tail()

In [ ]:
# Thong tin kieu du lieu va so luong gia tri khong null tung cot
df.info()

**Y nghia moi hang:**

Moi hang dai dien cho **mot tua game** tren Steam, duoc dinh danh duy nhat boi `app_id`.

**Co hang trung lap khong?**

In [ ]:
# Kiem tra hang trung lap hoan toan
n_dup_full = df.duplicated().sum()
print(f'Hang trung lap hoan toan  : {n_dup_full:,}')

# Kiem tra trung lap theo app_id (khoa chinh)
if 'app_id' in df.columns:
    n_dup_id = df.duplicated(subset='app_id').sum()
    print(f'app_id trung lap          : {n_dup_id:,}')
    if n_dup_id > 0:
        print('\nCac hang co app_id trung lap:')
        dup_mask = df.duplicated(subset='app_id', keep=False)
        display(df[dup_mask].sort_values('app_id').head(10))
else:
    print('Khong tim thay cot app_id — kiem tra lai ten cot thuc te')

### 2.2 Y nghia tung cot (Data Dictionary)

| Ten cot | Kieu du lieu | Y nghia | Gia tri dac biet / Ghi chu |
|---|---|---|---|
| `app_id` | int | ID dinh danh game tren Steam | Primary key |
| `name` | str | Ten tua game | |
| `release_date` | str | Ngay phat hanh | Can convert sang datetime |
| `developer` | str | Nha phat trien | Co the co nhieu nha phat trien, phan cach dau phay |
| `publisher` | str | Nha phat hanh | Tuong tu developer |
| `price` | float | Gia goc tinh bang USD | 0.0 = Free-to-play |
| `positive_reviews` | int | Tong so danh gia tich cuc | |
| `negative_reviews` | int | Tong so danh gia tieu cuc | |
| `tags` | str | Tag do nguoi dung gan | Dang list (JSON hoac chuoi phan cach), can parse |
| `genres` | str | The loai chinh (do Steam phan loai) | |
| `categories` | str | Danh muc (Single-player, Multi-player, ...) | |
| `supported_languages` | str | Cac ngon ngu ho tro | Dang list, can parse |
| `platforms` | str | Nen tang ho tro | Windows / Mac / Linux |
| `achievements` | int | So thanh tich trong game | 0 = khong co achievement |
| `dlc_count` | int | So DLC da phat hanh | 0 = khong co DLC |

> **Luu y:** Cap nhat bang nay theo dung ten cot thuc te cua file CSV sau khi load xong.

In [ ]:
# In tat ca ten cot thuc te de cap nhat bang Data Dictionary
print(f'Tong so cot: {df.shape[1]}')
print()
print(f'{'#':<5} {'Ten cot':<40} {'Kieu':>10}  {'Non-null':>10}  {'% missing':>10}')
print('-' * 80)
for i, col in enumerate(df.columns, 1):
    non_null = df[col].notna().sum()
    missing_pct = (1 - non_null / len(df)) * 100
    print(f'{i:<5} {col:<40} {str(df[col].dtype):>10}  {non_null:>10,}  {missing_pct:>9.1f}%')

### 2.3 Kieu du lieu — Co cot nao khong dung kieu khong?

In [ ]:
# Dem so luong cot theo kieu du lieu hien tai
print('Phan bo kieu du lieu:')
print(df.dtypes.value_counts())
print()

# Kiem tra cac kieu can chuyen doi
print('Cac van de ve kieu du lieu can xu ly:')
for col in df.columns:
    dtype = df[col].dtype
    # Phat hien cot so dang luu duoi dang object
    if dtype == object:
        sample = df[col].dropna().head(5).tolist()
        print(f'  [object] {col}: {sample}')

**Nhan xet ve kieu du lieu:**

*(Dien sau khi chay cell — liet ke cu the cot nao can convert va ly do)*

| Cot | Kieu hien tai | Kieu can chuyen | Ly do |
|---|---|---|---|
| `release_date` | object | datetime64 | De tinh toan ngay thang, loc theo nam |
| `price` | *(kiem tra)* | float | Dam bao co the tinh toan |
| `tags` | object | list | Can parse de phan tich tung tag |
| *(ten cot khac)* | ... | ... | ... |

### 2.4 Missing values

In [ ]:
# Tinh ti le missing (%) tung cot, sap xep giam dan
missing_count = df.isnull().sum()
missing_pct   = (missing_count / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'cot'      : missing_count.index,
    'so_null'  : missing_count.values,
    'phan_tram': missing_pct.values
}).query('so_null > 0').sort_values('phan_tram', ascending=False)

if missing_df.empty:
    print('Khong co missing values trong dataset.')
else:
    print(f'Co {len(missing_df)} cot co missing values:')
    display(missing_df.reset_index(drop=True))

In [ ]:
# Visualize missing values bang missingno
if not missing_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Bar chart: ti le non-null tung cot
    msno.bar(df, ax=axes[0], color='steelblue', fontsize=11)
    axes[0].set_title('Ti le du lieu co gia tri theo tung cot', fontsize=12)

    # Matrix: phan bo vi tri null
    msno.matrix(df, ax=axes[1], fontsize=11)
    axes[1].set_title('Vi tri missing values (trang = null)', fontsize=12)

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '00_missing_values.png'), dpi=150)
    plt.show()
else:
    print('Khong co gi de visualize (dataset khong co missing).')

**Nhan xet missing values:**

*(Dien sau khi chay)*

- Cot nao co nhieu missing nhat?
- Missing co ngau nhien (MCAR) hay co he thong (MAR/MNAR)?
- Ke hoach xu ly: xoa hang / dien gia tri thay the / giu nguyen?

### 2.5 Phan tich cot so (Numerical columns)

In [ ]:
# Thong ke mo ta tat ca cot so
num_cols = df.select_dtypes(include='number').columns.tolist()
print(f'Co {len(num_cols)} cot so: {num_cols}\n')
df[num_cols].describe().T.style.format('{:.2f}')

In [ ]:
# Vet chi tiet tung cot so: min, max, median, % outlier (IQR method)
print(f'{'Cot':<30} {'Min':>12} {'Max':>15} {'Median':>12} {'% outlier':>12}')
print('-' * 85)
for col in num_cols:
    data = df[col].dropna()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outlier = ((data < lower) | (data > upper)).sum()
    pct_outlier = n_outlier / len(data) * 100
    print(f'{col:<30} {data.min():>12.2f} {data.max():>15.2f} {data.median():>12.2f} {pct_outlier:>11.1f}%')

In [ ]:
# Histogram + KDE + duong median cho tung cot so quan trong
# Chi ve cac cot co y nghia thuc te, bo qua id
plot_cols = [c for c in ['price', 'positive_reviews', 'negative_reviews',
                          'achievements', 'dlc_count'] if c in df.columns]

if plot_cols:
    ncols = min(len(plot_cols), 3)
    nrows = (len(plot_cols) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
    axes = np.array(axes).flatten()

    for i, col in enumerate(plot_cols):
        data = df[col].dropna()
        ax = axes[i]
        ax.hist(data, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
        med = data.median()
        mean = data.mean()
        ax.axvline(med,  color='red',    linestyle='--', linewidth=1.5, label=f'Median = {med:.1f}')
        ax.axvline(mean, color='orange', linestyle=':',  linewidth=1.5, label=f'Mean   = {mean:.1f}')
        ax.set_title(col, fontsize=12)
        ax.set_xlabel('Gia tri')
        ax.set_ylabel('Tan suat')
        ax.legend(fontsize=9)

    # An truc thua
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Phan phoi cac cot so', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '00_num_histograms.png'), dpi=150)
    plt.show()

In [ ]:
# Boxplot de phat hien outlier ro rang hon
if plot_cols:
    fig, axes = plt.subplots(1, len(plot_cols), figsize=(4 * len(plot_cols), 5))
    if len(plot_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, plot_cols):
        ax.boxplot(
            df[col].dropna(),
            vert=True,
            patch_artist=True,
            boxprops=dict(facecolor='lightsteelblue', color='navy'),
            medianprops=dict(color='red', linewidth=2),
            flierprops=dict(marker='o', markersize=3, alpha=0.4, color='gray')
        )
        ax.set_title(col, fontsize=11)
        ax.set_xticklabels([col], fontsize=9)

    plt.suptitle('Boxplot — Phat hien outlier', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '00_num_boxplots.png'), dpi=150)
    plt.show()

In [ ]:
# Kham pha them: phan chia game mien phi vs tra phi
if 'price' in df.columns:
    free_count = (df['price'] == 0).sum()
    paid_count = (df['price'] > 0).sum()
    total = len(df)
    print(f'Free-to-play : {free_count:,} game ({free_count/total*100:.1f}%)')
    print(f'Tra phi      : {paid_count:,} game ({paid_count/total*100:.1f}%)')
    print()

    # Phan phoi gia cac game tra phi (log scale)
    paid = df[df['price'] > 0]['price']
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].hist(paid, bins=50, color='coral', edgecolor='white')
    axes[0].set_title('Phan phoi gia game (tra phi)')
    axes[0].set_xlabel('Gia (USD)')
    axes[0].set_ylabel('So game')

    axes[1].hist(paid, bins=50, color='coral', edgecolor='white', log=True)
    axes[1].set_title('Phan phoi gia game — log scale (tra phi)')
    axes[1].set_xlabel('Gia (USD)')
    axes[1].set_ylabel('So game (log)')

    for ax in axes:
        ax.axvline(paid.median(), color='navy', linestyle='--',
                   linewidth=1.5, label=f'Median = {paid.median():.1f} USD')
        ax.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '00_price_distribution.png'), dpi=150)
    plt.show()

In [ ]:
# Tinh ty le review tich cuc (review score) cho moi game
if 'positive_reviews' in df.columns and 'negative_reviews' in df.columns:
    df['total_reviews'] = df['positive_reviews'] + df['negative_reviews']
    # Chi tinh review score voi game co >= 10 danh gia (tranh ti le sai lech khi it danh gia)
    df['review_score'] = np.where(
        df['total_reviews'] >= 10,
        df['positive_reviews'] / df['total_reviews'] * 100,
        np.nan
    )

    print('Thong ke review_score (% danh gia tich cuc, game >= 10 danh gia):')
    print(df['review_score'].describe())

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df['review_score'].dropna(), bins=50, color='mediumseagreen', edgecolor='white')
    ax.set_title('Phan phoi Review Score (% tich cuc)')
    ax.set_xlabel('Review Score (%)')
    ax.set_ylabel('So game')
    ax.axvline(df['review_score'].median(), color='red', linestyle='--',
               linewidth=1.5, label=f"Median = {df['review_score'].median():.1f}%")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '00_review_score_dist.png'), dpi=150)
    plt.show()

**Nhan xet cot so:**

*(Dien sau khi chay — viet day du, cu the)*

| Cot | Min | Max | Median | Mean | % Outlier | Nhan xet |
|---|---|---|---|---|---|---|
| `price` | ? | ? | ? | ? | ? | *(phan phoi lech phai? co gia bao nhieu?)* |
| `positive_reviews` | ? | ? | ? | ? | ? | *(phan bo nhu the nao?)* |
| `negative_reviews` | ? | ? | ? | ? | ? | *(co bat thuong khong?)* |
| `achievements` | ? | ? | ? | ? | ? | *(0 nghia la khong co)* |
| `dlc_count` | ? | ? | ? | ? | ? | *(phan bo ra sao?)* |

### 2.6 Phan tich cot phan loai (Categorical columns)

In [ ]:
# Tong quan tat ca cot categorical
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Co {len(cat_cols)} cot categorical:\n')
print(f'{'Ten cot':<35} {'Unique':>8} {'Missing%':>10}  {'Vi du 3 gia tri'}')
print('-' * 90)
for col in cat_cols:
    n_unique = df[col].nunique()
    missing  = df[col].isnull().mean() * 100
    examples = df[col].dropna().unique()[:3].tolist()
    print(f'{col:<35} {n_unique:>8,} {missing:>9.1f}%  {examples}')

In [ ]:
# Phan tich: Top 15 genres
if 'genres' in df.columns:
    top_genres = df['genres'].value_counts().head(15)

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(top_genres.index, top_genres.values,
                   color='steelblue', edgecolor='white')
    ax.invert_yaxis()
    ax.set_title('Top 15 Genres tren Steam', fontsize=13)
    ax.set_xlabel('So luong game')

    # Gan nhan gia tri len thanh bar
    for bar, val in zip(bars, top_genres.values):
        ax.text(val + 50, bar.get_y() + bar.get_height() / 2,
                f'{val:,}', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '00_top_genres.png'), dpi=150)
    plt.show()
    print(f'Tong so genres khac nhau: {df["genres"].nunique()}')

In [ ]:
# Phan tich: Top 10 developers
if 'developer' in df.columns:
    top_devs = df['developer'].value_counts().head(10)

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(top_devs.index, top_devs.values,
                   color='coral', edgecolor='white')
    ax.invert_yaxis()
    ax.set_title('Top 10 Nha Phat Trien (so tua game)', fontsize=13)
    ax.set_xlabel('So game')

    for bar, val in zip(bars, top_devs.values):
        ax.text(val + 1, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '00_top_developers.png'), dpi=150)
    plt.show()

In [ ]:
# Phan tich: phan nen tang (Windows / Mac / Linux)
# Gia su cot platforms la chuoi nhu 'windows,mac,linux'
if 'platforms' in df.columns:
    platform_counts = {
        'Windows': df['platforms'].str.contains('windows', case=False, na=False).sum(),
        'Mac'    : df['platforms'].str.contains('mac',     case=False, na=False).sum(),
        'Linux'  : df['platforms'].str.contains('linux',   case=False, na=False).sum(),
    }
    total = len(df)
    print('Nen tang ho tro:')
    for plat, cnt in platform_counts.items():
        print(f'  {plat:<10}: {cnt:,} ({cnt/total*100:.1f}%)')

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(platform_counts.keys(), platform_counts.values(),
           color=['#4472C4', '#ED7D31', '#A9D18E'], edgecolor='white')
    ax.set_title('So game theo nen tang ho tro')
    ax.set_ylabel('So luong game')
    for i, (k, v) in enumerate(platform_counts.items()):
        ax.text(i, v + 100, f'{v:,}', ha='center', fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '00_platforms.png'), dpi=150)
    plt.show()

**Nhan xet cot categorical:**

*(Dien sau khi chay)*

- Genre pho bien nhat la gi? Chiem bao nhieu % tong game?
- Developer nao co nhieu game nhat? Co hien tuong mot developer co hang tram game khong?
- Nen tang nao duoc ho tro nhieu nhat? Linux va Mac chiem bao nhieu %?

### 2.7 Phan phoi theo thoi gian

In [ ]:
# Convert release_date sang datetime
if 'release_date' in df.columns:
    df['release_date_dt'] = pd.to_datetime(df['release_date'], errors='coerce')
    n_failed = df['release_date_dt'].isna().sum() - df['release_date'].isna().sum()
    print(f'Convert thanh cong: {df["release_date_dt"].notna().sum():,} hang')
    print(f'Convert that bai (sai dinh dang): {n_failed:,} hang')

    # Lay nam, thang
    df['release_year']  = df['release_date_dt'].dt.year
    df['release_month'] = df['release_date_dt'].dt.month

    # Khoang thoi gian cua du lieu
    print(f'\nNgay phat hanh som nhat: {df["release_date_dt"].min()}')
    print(f'Ngay phat hanh muon nhat: {df["release_date_dt"].max()}')

In [ ]:
# So game phat hanh theo nam (2000-2025)
if 'release_year' in df.columns:
    year_range = df[
        (df['release_year'] >= 2000) & (df['release_year'] <= 2025)
    ]['release_year'].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(year_range.index, year_range.values, color='steelblue', edgecolor='white')
    ax.set_title('So game phat hanh theo nam tren Steam (2000-2025)', fontsize=13)
    ax.set_xlabel('Nam')
    ax.set_ylabel('So luong game')
    ax.xaxis.set_major_locator(mticker.MultipleLocator(2))

    # Gan gia tri len cot nam co nhieu game nhat
    peak_year = year_range.idxmax()
    peak_val  = year_range.max()
    ax.annotate(f'Dinh: {peak_year}\n({peak_val:,} game)',
                xy=(peak_year, peak_val),
                xytext=(peak_year - 3, peak_val * 0.85),
                arrowprops=dict(arrowstyle='->', color='red'),
                fontsize=10, color='red')

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '00_games_per_year.png'), dpi=150)
    plt.show()

In [ ]:
# So game phat hanh theo thang (seasonality)
if 'release_month' in df.columns:
    month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec']
    month_counts = df['release_month'].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar([month_labels[m-1] for m in month_counts.index],
           month_counts.values, color='mediumpurple', edgecolor='white')
    ax.set_title('So game phat hanh theo thang (seasonality)', fontsize=13)
    ax.set_xlabel('Thang')
    ax.set_ylabel('So luong game')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '00_games_per_month.png'), dpi=150)
    plt.show()
    print('Thang co nhieu game phat hanh nhat:', month_labels[month_counts.idxmax() - 1])

**Nhan xet theo thoi gian:**

*(Dien sau khi chay)*

- Steam bat dau bung no ve so luong game tu nam nao?
- Co thang nao co nhieu game ra mat han (mua he, cuoi nam)?
- Co xu huong gi dang chu y trong nhung nam gan day?

### 2.8 Tuong quan giua cac bien so (Correlation)

In [ ]:
# Ma tran tuong quan Pearson giua cac cot so
corr_cols = [c for c in ['price', 'positive_reviews', 'negative_reviews',
                          'achievements', 'dlc_count', 'review_score', 'total_reviews']
             if c in df.columns]

if len(corr_cols) >= 2:
    corr_matrix = df[corr_cols].corr()

    fig, ax = plt.subplots(figsize=(10, 7))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # chi hien nua duoi
    sns.heatmap(
        corr_matrix,
        mask=mask,
        annot=True,
        fmt='.2f',
        cmap='coolwarm',
        center=0,
        linewidths=0.5,
        ax=ax,
        square=True
    )
    ax.set_title('Ma tran tuong quan (Pearson) — cac bien so', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '00_correlation_heatmap.png'), dpi=150)
    plt.show()

    # In cap co tuong quan cao (|r| >= 0.5)
    print('Cac cap bien co tuong quan cao (|r| >= 0.5):')
    for i in range(len(corr_cols)):
        for j in range(i+1, len(corr_cols)):
            r = corr_matrix.iloc[i, j]
            if abs(r) >= 0.5:
                print(f'  {corr_cols[i]} <-> {corr_cols[j]}: r = {r:.3f}')

**Nhan xet tuong quan:**

*(Dien sau khi chay)*

- Cap nao co tuong quan manh nhat? Thuan hay nghich?
- Gia co tuong quan voi so luong danh gia khong?
- Ket qua co phu hop voi truc giac khong?

---

## 3. Dat cau hoi co y nghia

> **Yeu cau de bai:** >= so thanh vien nhom cau hoi.
> Moi cau hoi phai:
> 1. **Co y nghia** — khong qua don gian (khong chi 1-2 dong code la ra)
> 2. **Neu ro loi ich** — ai huong loi khi biet cau tra loi?

### Cau hoi 1 — *(Tieu de cau hoi)*

**Cau hoi:** *(Viet cau hoi ro rang, cu the va co the do luong duoc)*

**Loi ich:** *(Tai sao cau hoi nay co gia tri? Vi du: giup nha phat trien dinh gia hop ly, giup nguoi choi tim game tot, giup du bao xu huong thi truong...)*

---

### Cau hoi 2 — *(Tieu de cau hoi)*

**Cau hoi:** *(Viet cau hoi ro rang, cu the va co the do luong duoc)*

**Loi ich:** *(Tai sao cau hoi nay co gia tri?)*

---

### Cau hoi 3 — *(Tieu de cau hoi)*

**Cau hoi:** *(Viet cau hoi ro rang, cu the va co the do luong duoc)*

**Loi ich:** *(Tai sao cau hoi nay co gia tri?)*

---

> *(Them cau hoi neu nhom co nhieu hon 3 thanh vien)*

---

## 4. Tien xu ly & Phan tich

> Voi moi cau hoi: **(1) Mo ta tien xu ly bang text** -> **(2) Code tien xu ly** -> **(3) Mo ta phan tich bang text** -> **(4) Code phan tich & visualization** -> **(5) Ket luan**

### 4.1 Tra loi Cau hoi 1 — *(Tieu de)*

#### 4.1.1 Tien xu ly

**Mo ta cac buoc tien xu ly** *(viet de nguoi doc hieu ma khong can doc code)*:

1. **Buoc 1:** *(Vi du: Loc bo cac game chua du so danh gia (< 10 danh gia) vi ti le review se khong tin cay)*
2. **Buoc 2:** *(Vi du: Xu ly missing values cua cot X bang cach dien Y hoac xoa hang)*
3. **Buoc 3:** *(Vi du: Tao them cot tinh toan moi de phuc vu phan tich)*
4. **Buoc 4 (neu co):** *(Vi du: Chuan hoa / normalize neu can so sanh cac don vi khac nhau)*

In [ ]:
# ── Tien xu ly cho Cau hoi 1 ─────────────────────────────────────────────────
df_q1 = df.copy()
print(f'Shape ban dau: {df_q1.shape}')

# --- Buoc 1: [Mo ta buoc 1] ---
# df_q1 = df_q1[...]
# print(f'Sau buoc 1: {df_q1.shape}')

# --- Buoc 2: [Mo ta buoc 2] ---
# df_q1['ten_cot'] = ...
# print(f'Sau buoc 2: {df_q1.shape}')

# --- Buoc 3: [Mo ta buoc 3] ---
# ...

print(f'Shape sau tien xu ly: {df_q1.shape}')
df_q1.head(3)  # kiem tra ket qua

#### 4.1.2 Phan tich

**Mo ta cach phan tich** *(viet de nguoi doc hieu y tuong ma khong can doc code)*:

*(Vi du: Nhom game theo the loai, tinh trung binh review_score cho moi nhom, sau do so sanh bang bieu do cot ngang de thay ro su khac biet.)*

In [ ]:
# ── Phan tich Cau hoi 1 ───────────────────────────────────────────────────────
# --- Buoc phan tich chinh ---
# result_q1 = df_q1.groupby('cot_nhom')[['cot_gia_tri']].agg(['mean', 'median', 'count'])
# result_q1.columns = ['_'.join(c) for c in result_q1.columns]
# result_q1 = result_q1.sort_values('mean_cot_gia_tri', ascending=False)
# result_q1.head(10)

In [ ]:
# ── Visualization Cau hoi 1 ──────────────────────────────────────────────────
# Chon loai bieu do phu hop:
#   - Bar chart  : so sanh cac nhom roi rac
#   - Line chart : xu huong theo thoi gian
#   - Scatter    : moi quan he giua 2 bien lien tuc
#   - Box plot   : phan phoi theo nhom
#   - Heatmap    : tuong quan ma tran

fig, ax = plt.subplots(figsize=(12, 5))

# --- Code ve bieu do tai day ---
# ax.bar(result_q1.index, result_q1['mean_cot_gia_tri'], color='steelblue')

ax.set_title('*(Tieu de bieu do)*', fontsize=13)
ax.set_xlabel('*(Nhan truc x)*')
ax.set_ylabel('*(Nhan truc y)*')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'q1_result.png'), dpi=150)
plt.show()

#### 4.1.3 Ket luan

**Cau tra loi cho cau hoi 1:**

*(Viet ket luan cu the dua tren so lieu thuc te — tranh viet chung chung)*

- Phat hien chinh: ...
- Con so cu the: ...
- Y nghia / giai thich: ...
- Han che cua phan tich nay (neu co): ...

---

### 4.2 Tra loi Cau hoi 2 — *(Tieu de)*

#### 4.2.1 Tien xu ly

**Mo ta cac buoc tien xu ly:**

1. **Buoc 1:** *(Mo ta)*
2. **Buoc 2:** *(Mo ta)*
3. **Buoc 3 (neu co):** *(Mo ta)*

In [ ]:
# ── Tien xu ly cho Cau hoi 2 ─────────────────────────────────────────────────
df_q2 = df.copy()
print(f'Shape ban dau: {df_q2.shape}')

# --- Buoc 1 ---
# ...

# --- Buoc 2 ---
# ...

print(f'Shape sau tien xu ly: {df_q2.shape}')
df_q2.head(3)

#### 4.2.2 Phan tich

**Mo ta cach phan tich:**

*(Mo ta y tuong phan tich)*

In [ ]:
# ── Phan tich Cau hoi 2 ───────────────────────────────────────────────────────
# TODO: code phan tich
# result_q2 = ...

In [ ]:
# ── Visualization Cau hoi 2 ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

# TODO: code ve bieu do

ax.set_title('*(Tieu de bieu do)*', fontsize=13)
ax.set_xlabel('*(Nhan truc x)*')
ax.set_ylabel('*(Nhan truc y)*')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'q2_result.png'), dpi=150)
plt.show()

#### 4.2.3 Ket luan

**Cau tra loi cho cau hoi 2:**

- Phat hien chinh: ...
- Con so cu the: ...
- Y nghia / giai thich: ...
- Han che cua phan tich nay (neu co): ...

---

### 4.3 Tra loi Cau hoi 3 — *(Tieu de)*

#### 4.3.1 Tien xu ly

**Mo ta cac buoc tien xu ly:**

1. **Buoc 1:** *(Mo ta)*
2. **Buoc 2:** *(Mo ta)*
3. **Buoc 3 (neu co):** *(Mo ta)*

In [ ]:
# ── Tien xu ly cho Cau hoi 3 ─────────────────────────────────────────────────
df_q3 = df.copy()
print(f'Shape ban dau: {df_q3.shape}')

# --- Buoc 1 ---
# ...

# --- Buoc 2 ---
# ...

print(f'Shape sau tien xu ly: {df_q3.shape}')
df_q3.head(3)

#### 4.3.2 Phan tich

**Mo ta cach phan tich:**

*(Mo ta y tuong phan tich)*

In [ ]:
# ── Phan tich Cau hoi 3 ───────────────────────────────────────────────────────
# TODO: code phan tich
# result_q3 = ...

In [ ]:
# ── Visualization Cau hoi 3 ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

# TODO: code ve bieu do

ax.set_title('*(Tieu de bieu do)*', fontsize=13)
ax.set_xlabel('*(Nhan truc x)*')
ax.set_ylabel('*(Nhan truc y)*')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'q3_result.png'), dpi=150)
plt.show()

#### 4.3.3 Ket luan

**Cau tra loi cho cau hoi 3:**

- Phat hien chinh: ...
- Con so cu the: ...
- Y nghia / giai thich: ...
- Han che cua phan tich nay (neu co): ...

---

## 5. Phan anh (Reflection)

### 5.1 Kho khan cua tung thanh vien

*(Moi thanh vien tu viet phan cua minh sau khi hoan thanh cong viec)*

***(Member 1) — Ho ten:***

*(Viet cu the: gap kho khan gi? O buoc nao? Giai quyet bang cach nao?)*

---

***(Member 2) — Ho ten:***

*(Viet cu the)*

---

***(Member 3) — Ho ten:***

*(Viet cu the)*

### 5.2 Bai hoc rut ra cua tung thanh vien

***(Member 1):***

*(Vi du: hoc duoc ky nang xu ly du lieu dang list trong cot, hoc duoc cach doc hieu ket qua correlation, hieu ro hon quy trinh EDA...)*

---

***(Member 2):***

*(Viet cu the)*

---

***(Member 3):***

*(Viet cu the)*

### 5.3 Neu co them thoi gian, nhom se lam gi?

*(Thao luan nhom — neu cu the, khong viet chung chung)*

- *(Vi du: Xay dung mo hinh du doan review score dua tren gia, genre, developer)*
- *(Vi du: So sanh du lieu nay voi bo du lieu Steam cac nam truoc de thay xu huong)*
- *(Vi du: Phan tich sau hon ve tags — parse va dem tan suat tung tag, tim tag nao lien quan den do pho bien cua game)*
- ...

---

## 6. Tai lieu tham khao

| # | Tai lieu | Link |
|---|---|---|
| 1 | Dataset: Steam Games (March 2025) | [Kaggle](https://www.kaggle.com/) |
| 2 | Steam Web API Documentation | [steamcommunity.com/dev](https://steamcommunity.com/dev) |
| 3 | Pandas documentation | [pandas.pydata.org](https://pandas.pydata.org/docs/) |
| 4 | Seaborn documentation | [seaborn.pydata.org](https://seaborn.pydata.org/) |
| 5 | Matplotlib documentation | [matplotlib.org](https://matplotlib.org/stable/) |
| 6 | Plotly documentation | [plotly.com/python](https://plotly.com/python/) |
| 7 | Missingno documentation | [github.com/ResidentMario/missingno](https://github.com/ResidentMario/missingno) |
| 8 | *(Them tai lieu khac da tham khao)* | *(link)* |

---

<p align="center">
  <sub>FIT@HCMUS - Khoa hoc Du lieu 4.0 - 2025</sub>
</p>